# UncondDiff — Results Notebook

Load a trained unconditional diffusion model (`DiffusionModel` with `conditioning='none'`, `encoding=False`)
and produce:
- Single-sample comparison view
- Per-channel MAE / RMSE / PSNR
- Train / validation loss curves
- Multifield physical-consistency metrics (from pre-run CSV)
- Overlay plots
- Full test-set statistics

No RRDB encoder is used — sampling is purely unconditional.

**Before running:** fill in the paths in the next cell.

In [ ]:
# ── USER CONFIG — edit these paths before running ─────────────────────────────

# UncondDiff run directory (train_srdiff.py --modeltype diffusion, conditioning=none, encoding=False)
DIFF_RUN_DIR   = "/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/direct/uncond_diffusion/cs_both_n3"
# Data root — same as root_folder in your config YAML
DATA_ROOT      = "/trace/group/forgelab/ngng/multifield/data_fields"
# Output dir from multifield_eval.py; set None to skip consistency-metric cells
EVAL_OUT_DIR   = "/trace/group/forgelab/ngng/multifield/eval_results"
# Label inside EVAL_OUT_DIR
EVAL_LABEL     = "UncondDiff_both"

# ── Dataset / model config — must match training YAML ─────────────────────────
FIELD_NAMES      = ['temperature', 'liqlabel']
N_STEPS          = 3
DOWNSCALE_METHOD = 'direct'
NORMALIZE        = 'standardize'
TIMESTEPS        = 1000
SCHEDULE         = 'linear'
DEVICE           = 'cuda'

# ── Visualisation config ───────────────────────────────────────────────────────
DATA_SPLIT     = 'test'
BATCH_INDEX    = 0
SAMPLE_INDEX   = 0
CHANNEL_INDEX  = 0
SAMPLER        = 'DDIM'
DDIM_SKIP      = 50
BATCH_SIZE     = 4
T_LIQ          = 1700.0
LIQ_THR        = 0.5
EXPORT_RESULTS = False

In [ ]:
%matplotlib inline
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')  # disable CUDA on login nodes; delete this line on a GPU node

from pathlib import Path
import sys
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import torch
from torch.utils.data import DataLoader
from scipy.ndimage import gaussian_filter
from IPython.display import display as _ipy_display

# Patch plt.show() so figures render in VS Code remote / headless Jupyter
_plt_show_orig = plt.show
def _show(*args, **kwargs):
    for num in plt.get_fignums():
        _ipy_display(plt.figure(num))
    plt.close('all')
plt.show = _show

if not torch.cuda.is_available() and DEVICE == 'cuda':
    print('No GPU found — falling back to CPU (inference will be slow for large test sets)')
    DEVICE = 'cpu'

def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'setup.py').exists() and (path / 'diffusionsr').exists():
            return path
    raise RuntimeError('Could not find project root — run from within the repo')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')

## Shared Helpers

In [ ]:
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.runners.plot_training_curves import collect_curves

def build_datasets(field_names=FIELD_NAMES, n_steps=N_STEPS):
    kw = dict(downscale_method=DOWNSCALE_METHOD, root_folder=DATA_ROOT,
              normalize=NORMALIZE, n_steps=n_steps, field_names=field_names)
    return (SimulationXZDataset(split='train', **kw),
            SimulationXZDataset(split='dev',   **kw),
            SimulationXZDataset(split='test',  **kw))

def get_batch(loader, batch_index):
    for i, batch in enumerate(loader):
        if i == batch_index: return batch
    raise IndexError(f'batch_index {batch_index} out of range')

def as_numpy(x):
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

def display_settings(field_name):
    if field_name == 'temperature': return 'jet', 293.0, 5000.0
    elif field_name == 'liqlabel':  return 'plasma', 0.0, 1.0
    return 'viridis', None, None

def mae_rmse(pred, target):
    p, t = np.asarray(pred).ravel(), np.asarray(target).ravel()
    return {'MAE': float(np.mean(np.abs(p - t))),
            'RMSE': float(np.sqrt(np.mean((p - t)**2)))}

def psnr_val(pred, target):
    mse = np.mean((np.asarray(pred) - np.asarray(target))**2)
    if mse == 0: return float('inf')
    return float(20 * np.log10(np.max(np.abs(np.asarray(target))) / np.sqrt(mse)))

from scipy.spatial import KDTree
from scipy.ndimage import binary_erosion

def boundary_pixels(mask):
    return np.argwhere(mask & ~binary_erosion(mask, structure=np.ones((3,3))))

def chamfer_dist(mask_a, mask_b):
    b_a, b_b = boundary_pixels(mask_a), boundary_pixels(mask_b)
    if len(b_a) == 0 or len(b_b) == 0: return float('nan')
    return float((KDTree(b_b).query(b_a)[0].mean() + KDTree(b_a).query(b_b)[0].mean()) / 2)

def consistency_metrics(bin_t, bin_liq):
    inter = (bin_t & bin_liq).sum(); union = (bin_t | bin_liq).sum()
    iou = inter / union if union > 0 else float('nan')
    mse = float(np.mean((bin_t.astype(float) - bin_liq.astype(float))**2))
    return iou, mse, chamfer_dist(bin_t, bin_liq)

def overlay_plot(T_bg, T_bin, liq_bin, title, ax=None, sigma=1.5):
    standalone = ax is None
    if standalone: fig, ax = plt.subplots(figsize=(5, 4), dpi=150)
    if T_bg is not None:
        ax.imshow(T_bg.T, origin='lower', cmap='jet', vmin=293, vmax=5000, aspect='auto')
    ax.contour(gaussian_filter(T_bin.T.astype(float), sigma), levels=[0.5],
               colors=['red'], linewidths=[1.5], origin='lower')
    ax.contour(gaussian_filter(liq_bin.T.astype(float), sigma), levels=[0.5],
               colors=['blue'], linewidths=[1.5], origin='lower')
    ax.legend(handles=[
        plt.Line2D([0],[0], color='red',  lw=1.5, label='Binary-T (T>1700K)'),
        plt.Line2D([0],[0], color='blue', lw=1.5, label='Liqlabel (liq>0.5)'),
    ], fontsize=7, loc='upper right')
    ax.set_title(title, fontsize=8); ax.axis('off')
    if standalone: plt.tight_layout(); plt.show()

## Load UncondDiff Model

No encoder — `encoding=False`, `conditioning='none'`.

In [ ]:
from diffusionsr.runners.train_diffusion import DiffusionModel

train_ds, dev_ds, test_ds = build_datasets()
print(f'Train: {len(train_ds)}  Dev: {len(dev_ds)}  Test: {len(test_ds)}')
print(f'Fields: {train_ds.field_names}  |  HR shape: {train_ds.img_shape}  |  {train_ds.factor}x')

model = DiffusionModel(
    results_folder=DIFF_RUN_DIR,
    lr_encoder_folder=DIFF_RUN_DIR,  # unused, but must be a valid path
    train_dataset=train_ds,
    dev_dataset=dev_ds,
    test_dataset=test_ds,
    timesteps=TIMESTEPS,
    conditioning='none',
    encoding=False,
    schedule=SCHEDULE,
    device=DEVICE,
    enc_output=False,
)
model.load_saved_model()
print(f'UncondDiff model loaded from {DIFF_RUN_DIR}')

## Select Sample and Run Inference

Unconditional generation: sampling produces a novel HR field from pure noise, independent of the LR input.

In [ ]:
loader = DataLoader(test_ds if DATA_SPLIT == 'test' else
                    dev_ds  if DATA_SPLIT == 'dev'  else train_ds,
                    batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
batch = get_batch(loader, BATCH_INDEX)
res, hr, true_lr, upscaled_lr = batch[:4]

hr_s  = hr[SAMPLE_INDEX:SAMPLE_INDEX+1]
ul_s  = upscaled_lr[SAMPLE_INDEX:SAMPLE_INDEX+1]
lr_s  = true_lr[SAMPLE_INDEX:SAMPLE_INDEX+1]

# No conditioning — x_e=None
with torch.no_grad():
    if SAMPLER == 'DDPM':
        samples = model.batch_sample(dataset=test_ds, batch=hr_s.to(DEVICE),
                                     x_e=None, sampler='DDPM')
    else:
        samples = model.batch_sample(dataset=test_ds, batch=hr_s.to(DEVICE),
                                     x_e=None, sampler='DDIM', skip=DDIM_SKIP)

fn = test_ds.field_names
pred_phys = test_ds.unscale_data(samples[-1].cpu().numpy()[0], input_type='hr')
hr_phys   = test_ds.unscale_data(as_numpy(hr_s[0]),            input_type='hr')
lr_phys   = test_ds.unscale_data(as_numpy(lr_s[0]),            input_type='lr')
up_phys   = test_ds.unscale_data(as_numpy(ul_s[0]),            input_type='upscaled_lr')
print(f'Inference done. pred shape: {pred_phys.shape}')

## Comparison View

> **Note:** UncondDiff generates from pure noise — there is no guarantee the prediction matches the specific GT sample. Aggregate test-set statistics in the final section are more meaningful than single-sample comparison.

In [ ]:
ch = CHANNEL_INDEX
field = fn[ch] if ch < len(fn) else f'ch{ch}'
cmap, vmin, vmax = display_settings(field)

panels = [
    ('True LR',         lr_phys[ch]),
    ('Upscaled LR',     up_phys[ch]),
    ('Ground Truth HR', hr_phys[ch]),
    ('UncondDiff',      pred_phys[ch]),
]
fig, axes = plt.subplots(1, len(panels), figsize=(4.5*len(panels), 4), dpi=150)
for ax, (title, data) in zip(axes, panels):
    im = ax.imshow(data.T, origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_title(title, fontsize=10); ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=0.8)
fig.suptitle(f'UncondDiff  |  {field}  |  {DATA_SPLIT} b{BATCH_INDEX} s{SAMPLE_INDEX}', fontsize=10)
plt.tight_layout(); plt.show()

## Single-Sample Metrics

In [ ]:
import pandas as pd

rows = []
for i, fname in enumerate(fn):
    if i >= pred_phys.shape[0]: break
    m = mae_rmse(pred_phys[i], hr_phys[i]); m['PSNR'] = psnr_val(pred_phys[i], hr_phys[i]); m['field'] = fname
    bl = mae_rmse(up_phys[i], hr_phys[i]);  bl['PSNR'] = psnr_val(up_phys[i], hr_phys[i]);  bl['field'] = fname + ' (upscaled LR)'
    rows += [m, bl]
print(pd.DataFrame(rows)[['field','MAE','RMSE','PSNR']].to_string(index=False))

## Multifield Consistency — Single Sample

In [ ]:
has_T    = 'temperature' in fn
has_sdf  = 'sdfliqlabel' in fn
has_liq  = 'liqlabel' in fn or has_sdf

def _liq_mask(phys_arr):
    if has_sdf:
        return phys_arr[fn.index('sdfliqlabel')] < 0
    return phys_arr[fn.index('liqlabel')] > LIQ_THR

if has_T and has_liq:
    T_pred   = pred_phys[fn.index('temperature')]
    liq_pred = _liq_mask(pred_phys)
    T_gt     = hr_phys[fn.index('temperature')]
    liq_gt   = _liq_mask(hr_phys)

    iou_p, mse_p, cham_p = consistency_metrics(T_pred > T_LIQ, liq_pred)
    iou_g, mse_g, cham_g = consistency_metrics(T_gt   > T_LIQ, liq_gt)
    print(f'Predicted: IOU={iou_p:.4f}  MSE={mse_p:.4f}  Chamfer={cham_p:.2f} px')
    print(f'GT ref:    IOU={iou_g:.4f}  MSE={mse_g:.4f}  Chamfer={cham_g:.2f} px')
    print(f'Liquid field: {"sdfliqlabel (SDF<0)" if has_sdf else "liqlabel (>LIQ_THR)"}')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=150)
    overlay_plot(T_pred, T_pred > T_LIQ, liq_pred,
                 f'UncondDiff (IOU={iou_p:.3f})', ax=axes[0])
    overlay_plot(T_gt, T_gt > T_LIQ, liq_gt,
                 f'GT reference (IOU={iou_g:.3f})', ax=axes[1])
    plt.tight_layout(); plt.show()
else:
    print(f'Consistency metrics need temperature + a liquid field. Present: {fn}')


## Training Curves

In [ ]:
diff_curves = collect_curves(Path(DIFF_RUN_DIR))

if diff_curves:
    fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
    colors = plt.cm.tab10.colors
    for i, (label, values) in enumerate(diff_curves):
        ls = '--' if ('val' in label.lower() or 'test' in label.lower()) else '-'
        ax.plot(np.arange(1, len(values)+1), values, ls=ls,
                color=colors[i % len(colors)], label=label, lw=1.6, alpha=0.9)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('UncondDiff — Training Curves')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No loss files found in', DIFF_RUN_DIR)

### W&B Training Curves (Optional Fallback)

In [ ]:
WANDB_PROJECT = 'Flow3D_SuperResolution'
WANDB_RUN_ID  = None

if WANDB_RUN_ID is not None:
    try:
        import wandb
        api = wandb.Api()
        run = api.run(f"{os.environ.get('WANDB_ENTITY', 'ngng-')}/{WANDB_PROJECT}/{WANDB_RUN_ID}")
        history = run.history(keys=['train_loss', 'val_loss'], pandas=True)
        fig, ax = plt.subplots(figsize=(9, 4), dpi=150)
        if 'train_loss' in history: ax.plot(history['train_loss'].values, label='Train loss')
        if 'val_loss'   in history: ax.plot(history['val_loss'].values,   label='Val loss', ls='--')
        ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.set_title(f'W&B run {WANDB_RUN_ID}')
        ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
    except Exception as e:
        print(f'W&B query failed: {e}')
else:
    print('Set WANDB_RUN_ID to use this cell.')

## Training Config Comparison

Side-by-side comparison of key hyperparameters across all four model types trained on the multifield (both fields, n_steps=3) setting. Yellow rows highlight parameters that differ between models — all other rows are held constant, isolating the architectural variable under study.

In [ ]:
import yaml
import pandas as pd

_cfg_dir = PROJECT_ROOT / 'diffusionsr' / 'configs' / 'multifield'

_model_configs = {
    'DiffusionSR':  _cfg_dir / 'diffusionsr_both_n3.yml',
    'FlowMatching': _cfg_dir / 'flowmatch_both_n3.yml',
    'UncondDiff':   _cfg_dir / 'uncond_both_n3.yml',
    'LDM':          _cfg_dir / 'ldm_both_n3.yml',
}

_KEYS = [
    'epochs', 'batch_size', 'learning_rate', 'loss_type',
    'schedule', 'timesteps', 'conditioning', 'encoding',
    'n_steps', 'fields', 'normalize_method', 'downscale_method',
    'fm_n_steps', 'vae_epochs',
]

_rows = {}
for model_name, cfg_path in _model_configs.items():
    if cfg_path.exists():
        with open(cfg_path) as _f:
            _cfg = yaml.safe_load(_f)
        _rows[model_name] = {k: _cfg.get(k, '—') for k in _KEYS}
    else:
        _rows[model_name] = {k: f'(file not found: {cfg_path.name})' for k in _KEYS}

_df = pd.DataFrame(_rows, index=_KEYS)

def _highlight_diffs(row):
    vals = [v for v in row if str(v) != '—']
    differs = len(set(str(v) for v in vals)) > 1
    bg = 'background-color: #fff3cd' if differs else ''
    return [bg] * len(row)

print('Training Config Comparison — multifield (both fields, n_steps=3)')
print('Fields used in each notebook may differ from the "both" config shown; configs are the canonical source of truth.')
print('Yellow rows: parameters that differ between models.\n')
try:
    display(_df.style.apply(_highlight_diffs, axis=1).set_caption(
        'Key training hyperparameters per model type (multifield, both fields, n_steps=3)'))
except Exception:
    print(_df.to_string())


## Multifield Consistency — Full Test Set (from CSV)

In [ ]:
if EVAL_OUT_DIR is None:
    print('EVAL_OUT_DIR not set — skipping.')
else:
    exp_dir  = Path(EVAL_OUT_DIR) / EVAL_LABEL
    csv_path = exp_dir / 'per_sample.csv'
    sum_path = Path(EVAL_OUT_DIR) / 'consistency_summary.csv'

    if not csv_path.exists():
        print(f'per_sample.csv not found: {csv_path}\nRun multifield_eval.py first.')
    else:
        per_sample = pd.read_csv(csv_path)
        print(f'Loaded {len(per_sample)} samples')
        print(per_sample[['iou_pred','mse_pred','cham_pred','iou_gt','mse_gt','cham_gt']].describe().round(4))

        fig, axes = plt.subplots(1, 3, figsize=(13, 4), dpi=150)
        for ax, col, title in zip(axes,
                ['iou_pred','mse_pred','cham_pred'],
                ['IOU (↑)', 'MSE (↓)', 'Chamfer px (↓)']):
            valid = per_sample[col].dropna()
            ax.hist(valid, bins=30, color='mediumpurple', alpha=0.8)
            if col.replace('_pred','_gt') in per_sample.columns:
                gt_val = per_sample[col.replace('_pred','_gt')].dropna().mean()
                ax.axvline(gt_val, color='red', ls='--', lw=1.5, label=f'GT mean={gt_val:.3f}')
                ax.legend(fontsize=8)
            ax.set_title(title, fontsize=9)
        fig.suptitle(f'Per-sample distributions — {EVAL_LABEL}', fontsize=10)
        plt.tight_layout(); plt.show()

## Overlay Plots

In [ ]:
if EVAL_OUT_DIR is not None:
    overlay_dir = Path(EVAL_OUT_DIR) / EVAL_LABEL / 'overlays'
    pred_pngs = sorted(overlay_dir.glob('*.png')) if overlay_dir.exists() else []
    gt_dir    = Path(EVAL_OUT_DIR) / 'gt_overlays'
    gt_pngs   = sorted(gt_dir.glob('*.png')) if gt_dir.exists() else []
    if pred_pngs:
        n = min(len(pred_pngs), 5)
        fig, axes = plt.subplots(2, n, figsize=(4.5*n, 8), dpi=120)
        for j, (pf, gf) in enumerate(zip(pred_pngs[:n],
                                         gt_pngs[:n] if gt_pngs else [None]*n)):
            axes[0,j].imshow(mpimg.imread(str(pf))); axes[0,j].axis('off')
            axes[0,j].set_title(f'Pred {pf.stem}', fontsize=7)
            if gf and Path(gf).exists():
                axes[1,j].imshow(mpimg.imread(str(gf))); axes[1,j].axis('off')
                axes[1,j].set_title(f'GT {Path(gf).stem}', fontsize=7)
            else:
                axes[1,j].axis('off')
        fig.suptitle(f'Overlay plots — {EVAL_LABEL}', fontsize=10)
        plt.tight_layout(); plt.show()
    else:
        print(f'No overlay PNGs found. Run multifield_eval.py --mode overlays first.')

In [ ]:
from diffusionsr.analysis.analysis_functions import get_profile
import time

ANALYSIS_MAX_BATCH = None
ANALYSIS_CH        = 0
MELT_THRESHOLD     = 1900.0

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
maes, rmses = [], []
mp_profile_errors, kh_profile_errors = [], []
vc_maes = []
iou_list, mse_list, cham_list = [], [], []
iou_gt_list, mse_gt_list, cham_gt_list = [], [], []

t0 = time.perf_counter()
n_samples = 0

for i, batch in enumerate(test_loader):
    if ANALYSIS_MAX_BATCH is not None and i >= ANALYSIS_MAX_BATCH: break
    res_b, hr_b, lr_b, ul_b = batch[:4]
    with torch.no_grad():
        samps = model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE),
                                   x_e=None, sampler='DDPM')
    n_samples += hr_b.shape[0]
    for s in range(hr_b.shape[0]):
        p = test_ds.unscale_data(samps[-1].cpu().numpy()[s], input_type='hr')
        g = test_ds.unscale_data(as_numpy(hr_b[s]),          input_type='hr')
        maes.append(mae_rmse(p[ANALYSIS_CH], g[ANALYSIS_CH])['MAE'])
        rmses.append(mae_rmse(p[ANALYSIS_CH], g[ANALYSIS_CH])['RMSE'])
        try:
            pp_mp, pp_kh = get_profile(p[ANALYSIS_CH:ANALYSIS_CH+1])
            gp_mp, gp_kh = get_profile(g[ANALYSIS_CH:ANALYSIS_CH+1])
            mp_profile_errors.append(float(np.mean(np.abs(pp_mp - gp_mp))))
            kh_profile_errors.append(float(np.mean(np.abs(pp_kh - gp_kh))))
        except Exception: pass
        pred_mask   = p[ANALYSIS_CH] > MELT_THRESHOLD
        target_mask = g[ANALYSIS_CH] > MELT_THRESHOLD
        vc_maes.append(float(np.mean(np.abs(pred_mask.astype(float) - target_mask.astype(float)))))
        if has_T and has_liq:
            T_p  = p[fn.index('temperature')]; liq_p = _liq_mask(p)
            T_g  = g[fn.index('temperature')]; liq_g = _liq_mask(g)
            iou, mse_c, cham = consistency_metrics(T_p > T_LIQ, liq_p)
            iou_gt, mse_gt_c, cham_gt = consistency_metrics(T_g > T_LIQ, liq_g)
            iou_list.append(iou); mse_list.append(mse_c); cham_list.append(cham)
            iou_gt_list.append(iou_gt); mse_gt_list.append(mse_gt_c); cham_gt_list.append(cham_gt)

elapsed = time.perf_counter() - t0
speed   = n_samples / elapsed if elapsed > 0 else float('inf')

print(f'Test-set results (n={len(maes)}, DDPM sampler):')
print(f'  MAE  = {np.nanmean(maes):.4f} ± {np.nanstd(maes):.4f}')
print(f'  RMSE = {np.nanmean(rmses):.4f} ± {np.nanstd(rmses):.4f}')
if mp_profile_errors:
    print(f'  MP-MAE (melt pool depth)  = {np.nanmean(mp_profile_errors):.2f} ± {np.nanstd(mp_profile_errors):.2f} px')
    print(f'  KH-MAE (keyhole depth)    = {np.nanmean(kh_profile_errors):.2f} ± {np.nanstd(kh_profile_errors):.2f} px')
if vc_maes:
    print(f'  VC-MAE (vapor cavity mask)= {np.nanmean(vc_maes):.4f} ± {np.nanstd(vc_maes):.4f}')
if iou_list:
    print(f'  IOU pred     = {np.nanmean(iou_list):.4f} ± {np.nanstd(iou_list):.4f}')
    print(f'  IOU GT       = {np.nanmean(iou_gt_list):.4f} ± {np.nanstd(iou_gt_list):.4f}')
    print(f'  Chamfer pred = {np.nanmean(cham_list):.2f} ± {np.nanstd(cham_list):.2f} px')
print(f'\nInference speed: {speed:.2f} samples/s  ({elapsed:.1f}s for {n_samples} samples)')


## Result Plots — Fig. 4 / 10 / 11 / A.13 Style

Four plot types mirroring the paper figures:
- **Multi-sample grid** (Fig. 4): N=5 test inputs × [LR Input | Upscaled LR | UncondDiff | GT] (no encoder column), temperature field with liquid boundary (white SDF=0 contour).
- **Depth profiles** (Fig. 10): mean ± 1σ melt-pool and keyhole depth profiles across N=5 stochastic DDPM samples per test input.
- **Error histograms** (Fig. 11): per-sample MP-MAE and KH-MAE distributions vs. upscaled LR baseline. Requires test-set stats cell to have run first.
- **Sampler ablation** (Fig. A.13): DDIM MAE and MP-MAE vs. number of denoising steps.

In [ ]:
from scipy.ndimage import gaussian_filter as _gf
from diffusionsr.analysis.analysis_functions import get_profile

# ─── Multi-Sample Grid (Fig. 4 style, no encoder) ─────────────────────────────
N_GRID = 5
_gl = DataLoader(test_ds, batch_size=N_GRID, shuffle=False, drop_last=True)
_gb = next(iter(_gl))
_res_g, _hr_g, _lr_g, _ul_g = _gb[:4]
with torch.no_grad():
    _sg = model.batch_sample(dataset=test_ds, batch=_hr_g.to(DEVICE), x_e=None, sampler='DDPM')

def _mk_contour(ax, phys, sigma=1.5):
    if not (has_T and has_liq): return
    try:
        _lq = _liq_mask(phys)
        ax.contour(_gf(_lq.T.astype(float), sigma), levels=[0.5],
                   colors=['white'], linewidths=[1.], origin='lower', alpha=0.85)
    except Exception: pass

_cols10 = ['LR Input', 'Upscaled LR', 'UncondDiff', 'GT']
fig, axes = plt.subplots(N_GRID, 4, figsize=(16, 3.5*N_GRID), dpi=100)
if N_GRID == 1: axes = axes[np.newaxis]
for _r in range(N_GRID):
    _p  = test_ds.unscale_data(_sg[-1].cpu().numpy()[_r], input_type='hr')
    _g  = test_ds.unscale_data(as_numpy(_hr_g[_r]),       input_type='hr')
    _lr = test_ds.unscale_data(as_numpy(_lr_g[_r]),       input_type='lr')
    _ul = test_ds.unscale_data(as_numpy(_ul_g[_r]),       input_type='upscaled_lr')
    for _c, (_data, _phys) in enumerate(zip([_lr[0],_ul[0],_p[0],_g[0]], [_lr,_ul,_p,_g])):
        _ax = axes[_r, _c]
        _ax.imshow(_data.T, origin='lower', cmap='jet', vmin=293, vmax=5000, aspect='auto')
        _mk_contour(_ax, _phys)
        _ax.axis('off')
        if _r == 0: _ax.set_title(_cols10[_c], fontsize=9, fontweight='bold')
plt.suptitle('Multi-sample grid — temperature field + liquid boundary (white contour, DDPM)', fontsize=10)
plt.tight_layout(); plt.show()

# ─── Depth Profiles (Fig. 10 style, ±1σ from N stochastic DDPM samples) ───────
N_PROF_IN = 10; N_PROF_S = 5
_pl = DataLoader(test_ds, batch_size=1, shuffle=False, drop_last=False)
_mp_preds, _kh_preds, _mp_gt, _kh_gt, _xax = [], [], [], [], None
for _pi, _pb in enumerate(_pl):
    if _pi >= N_PROF_IN: break
    _, _hr_pb, _lr_pb, _ul_pb = _pb[:4]
    _gphy = test_ds.unscale_data(as_numpy(_hr_pb[0]), input_type='hr')
    try:
        _gmp, _gkh = get_profile(_gphy[ANALYSIS_CH:ANALYSIS_CH+1])
        _mp_gt.append(_gmp); _kh_gt.append(_gkh)
        if _xax is None: _xax = np.arange(len(_gmp))
    except Exception: continue
    _mps, _khs = [], []
    for _ in range(N_PROF_S):
        with torch.no_grad():
            _ss = model.batch_sample(dataset=test_ds, batch=_hr_pb.to(DEVICE), x_e=None, sampler='DDPM')
        _pphy = test_ds.unscale_data(_ss[-1].cpu().numpy()[0], input_type='hr')
        try:
            _pmp, _pkh = get_profile(_pphy[ANALYSIS_CH:ANALYSIS_CH+1])
            _mps.append(_pmp); _khs.append(_pkh)
        except Exception: pass
    if _mps: _mp_preds.append(np.stack(_mps)); _kh_preds.append(np.stack(_khs))

if _mp_gt and _mp_preds:
    _mp_gt_a = np.stack(_mp_gt); _mp_pd_a = np.concatenate(_mp_preds)
    _kh_gt_a = np.stack(_kh_gt); _kh_pd_a = np.concatenate(_kh_preds)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
    for _ax, _gt_a, _pd_a, _ttl, _col in [
            (ax1, _mp_gt_a, _mp_pd_a, 'Melt-Pool Depth Profile', 'mediumpurple'),
            (ax2, _kh_gt_a, _kh_pd_a, 'Keyhole Depth Profile',   'teal')]:
        _gm, _gs = _gt_a.mean(0), _gt_a.std(0)
        _pm, _ps = _pd_a.mean(0), _pd_a.std(0)
        _ax.plot(_xax, _pm, color=_col, lw=2, label='UncondDiff (mean)')
        _ax.fill_between(_xax, _pm-_ps, _pm+_ps, alpha=0.3, color=_col, label='±1σ')
        _ax.plot(_xax, _gm, 'k--', lw=1.5, label='GT (mean)')
        _ax.fill_between(_xax, _gm-_gs, _gm+_gs, alpha=0.15, color='black')
        _ax.set_xlabel('x (pixels)'); _ax.set_ylabel('Depth (pixels)')
        _ax.set_title(_ttl); _ax.legend(fontsize=9); _ax.grid(True, alpha=0.3)
    plt.suptitle(f'Depth profiles (mean ±1σ, N={N_PROF_S} stochastic, {N_PROF_IN} test inputs)', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('No depth profile data — confirm get_profile is compatible with output shape.')

# ─── Error Histograms (Fig. 11 style) ─────────────────────────────────────────
if 'mp_profile_errors' in dir() and mp_profile_errors and 'kh_profile_errors' in dir() and kh_profile_errors:
    _bl_mp, _bl_kh = [], []
    for _bb in DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False):
        _, _hr_bb, _, _ul_bb = _bb[:4]
        for _si in range(_hr_bb.shape[0]):
            _ulp = test_ds.unscale_data(as_numpy(_ul_bb[_si]), input_type='upscaled_lr')
            _hrp = test_ds.unscale_data(as_numpy(_hr_bb[_si]), input_type='hr')
            try:
                _bmp, _bkh = get_profile(_ulp[ANALYSIS_CH:ANALYSIS_CH+1])
                _gmp2, _gkh2 = get_profile(_hrp[ANALYSIS_CH:ANALYSIS_CH+1])
                _bl_mp.append(float(np.mean(np.abs(_bmp - _gmp2))))
                _bl_kh.append(float(np.mean(np.abs(_bkh - _gkh2))))
            except Exception: pass
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)
    for _ax, _sr, _bl, _ttl in zip(axes,
            [mp_profile_errors, kh_profile_errors], [_bl_mp, _bl_kh],
            ['MP Depth Error (melt-pool surface)', 'KH Depth Error (keyhole)']):
        _bmax = max(max(_sr+[0.001]), max(_bl+[0.001])) * 1.1
        _bins = np.linspace(0, _bmax, 35)
        _ax.hist(_sr, bins=_bins, alpha=0.75, color='mediumpurple',
                 label=f'UncondDiff  μ={np.mean(_sr):.2f}px')
        if _bl: _ax.hist(_bl, bins=_bins, alpha=0.6, color='gray',
                 label=f'Upscaled LR  μ={np.mean(_bl):.2f}px')
        _ax.axvline(np.mean(_sr), color='mediumpurple', ls='--', lw=2)
        if _bl: _ax.axvline(np.mean(_bl), color='gray', ls='--', lw=2)
        _ax.set_xlabel('MAE (pixels)'); _ax.set_ylabel('Count')
        _ax.set_title(_ttl, fontsize=9); _ax.legend(fontsize=8); _ax.grid(True, alpha=0.3)
    plt.suptitle('Error histograms: UncondDiff vs. upscaled LR baseline', fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print('Run test-set stats cell first (mp_profile_errors / kh_profile_errors required).')

# ─── Sampler Step Ablation (Fig. A.13 style) ──────────────────────────────────
_ABL_SKIPS = [1, 5, 10, 20, 50]; _ABL_N = 4
_ab_maes = {sk: [] for sk in _ABL_SKIPS}; _ab_mp = {sk: [] for sk in _ABL_SKIPS}
for _ai, _ab in enumerate(DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)):
    if _ai >= _ABL_N: break
    _, _abl_hr, _abl_lr, _abl_ul = _ab[:4]
    for _sk in _ABL_SKIPS:
        with torch.no_grad():
            _as = model.batch_sample(dataset=test_ds, batch=_abl_hr.to(DEVICE),
                                     x_e=None, sampler='DDIM', skip=_sk)
        for _s in range(_abl_hr.shape[0]):
            _pp = test_ds.unscale_data(_as[-1].cpu().numpy()[_s], input_type='hr')
            _gg = test_ds.unscale_data(as_numpy(_abl_hr[_s]), input_type='hr')
            _ab_maes[_sk].append(mae_rmse(_pp[ANALYSIS_CH], _gg[ANALYSIS_CH])['MAE'])
            try:
                _pmp, _ = get_profile(_pp[ANALYSIS_CH:ANALYSIS_CH+1])
                _gmp, _ = get_profile(_gg[ANALYSIS_CH:ANALYSIS_CH+1])
                _ab_mp[_sk].append(float(np.mean(np.abs(_pmp - _gmp))))
            except Exception: pass
_steps10 = [TIMESTEPS // sk for sk in _ABL_SKIPS]
_mae_v   = [np.nanmean(_ab_maes[sk]) for sk in _ABL_SKIPS]
_mp_v    = [np.nanmean(_ab_mp[sk]) if _ab_mp[sk] else float('nan') for sk in _ABL_SKIPS]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
ax1.plot(_steps10, _mae_v, 'o-', color='mediumpurple', lw=2, ms=8)
ax1.set_xlabel('DDIM denoising steps'); ax1.set_ylabel('MAE (temperature field)')
ax1.set_title('DDIM — Field MAE vs. Steps'); ax1.invert_xaxis(); ax1.grid(True, alpha=0.3)
ax2.plot(_steps10, _mp_v, 's-', color='teal', lw=2, ms=8)
ax2.set_xlabel('DDIM denoising steps'); ax2.set_ylabel('MP-MAE (pixels)')
ax2.set_title('DDIM — MP Depth MAE vs. Steps'); ax2.invert_xaxis(); ax2.grid(True, alpha=0.3)
plt.suptitle(f'Sampler step ablation — DDIM (skip={_ABL_SKIPS} → {_steps10[::-1]} steps)', fontsize=11)
plt.tight_layout(); plt.show()


## Full Test-Set Statistics

In [ ]:
from diffusionsr.analysis.analysis_functions import get_profile

ANALYSIS_MAX_BATCH = None
ANALYSIS_CH        = 0
MELT_THRESHOLD     = 1900.0

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
maes, rmses, profile_maes = [], [], []
iou_list, mse_list, cham_list = [], [], []
iou_gt_list, mse_gt_list, cham_gt_list = [], [], []

for i, batch in enumerate(test_loader):
    if ANALYSIS_MAX_BATCH is not None and i >= ANALYSIS_MAX_BATCH: break
    res_b, hr_b, lr_b, ul_b = batch[:4]
    with torch.no_grad():
        samps = model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE),
                                   x_e=None, sampler='DDIM', skip=DDIM_SKIP)
    for s in range(hr_b.shape[0]):
        p = test_ds.unscale_data(samps[-1].cpu().numpy()[s], input_type='hr')
        g = test_ds.unscale_data(as_numpy(hr_b[s]),          input_type='hr')
        m = mae_rmse(p[ANALYSIS_CH], g[ANALYSIS_CH])
        maes.append(m['MAE']); rmses.append(m['RMSE'])
        try:
            pp = get_profile(p[ANALYSIS_CH], threshold=MELT_THRESHOLD)
            gp = get_profile(g[ANALYSIS_CH], threshold=MELT_THRESHOLD)
            if pp is not None and gp is not None:
                profile_maes.append(float(np.mean(np.abs(np.array(pp) - np.array(gp)))))
        except Exception: pass
        if has_T and has_liq:
            T_p, liq_p = p[fn.index('temperature')], p[fn.index('liqlabel')]
            T_g_c, liq_g_c = g[fn.index('temperature')], g[fn.index('liqlabel')]
            iou, mse_c, cham = consistency_metrics(T_p > T_LIQ, liq_p > LIQ_THR)
            iou_gt, mse_gt_c, cham_gt = consistency_metrics(T_g_c > T_LIQ, liq_g_c > LIQ_THR)
            iou_list.append(iou); mse_list.append(mse_c); cham_list.append(cham)
            iou_gt_list.append(iou_gt); mse_gt_list.append(mse_gt_c); cham_gt_list.append(cham_gt)

print(f'Test-set results (n={len(maes)}):')
print(f'  MAE  = {np.nanmean(maes):.4f} ± {np.nanstd(maes):.4f}')
print(f'  RMSE = {np.nanmean(rmses):.4f} ± {np.nanstd(rmses):.4f}')
if profile_maes:
    print(f'  Profile MAE = {np.nanmean(profile_maes):.4f} ± {np.nanstd(profile_maes):.4f}')
if iou_list:
    print(f'  IOU pred     = {np.nanmean(iou_list):.4f} ± {np.nanstd(iou_list):.4f}')
    print(f'  IOU GT       = {np.nanmean(iou_gt_list):.4f} ± {np.nanstd(iou_gt_list):.4f}  (GT self-consistency ceiling)')
    print(f'  MSE pred     = {np.nanmean(mse_list):.4f} ± {np.nanstd(mse_list):.4f}')
    print(f'  MSE GT       = {np.nanmean(mse_gt_list):.4f} ± {np.nanstd(mse_gt_list):.4f}')
    print(f'  Chamfer pred = {np.nanmean(cham_list):.2f} ± {np.nanstd(cham_list):.2f} px')
    print(f'  Chamfer GT   = {np.nanmean(cham_gt_list):.2f} ± {np.nanstd(cham_gt_list):.2f} px')

## Export

In [ ]:
if EXPORT_RESULTS:
    out = Path(DIFF_RUN_DIR) / f'results_{DATA_SPLIT}_b{BATCH_INDEX}_s{SAMPLE_INDEX}.npz'
    np.savez(str(out), true_lr=lr_phys, upscaled_lr=up_phys,
             ground_truth=hr_phys, prediction=pred_phys)
    print(f'Saved → {out}')
else:
    print('Set EXPORT_RESULTS = True to save.')